This code reads and preprocesses the Gene expression data from MelanoDB

Some configurations

In [ ]:
DROP_NULLS = True
DROP_QPCR = True             
SELECT_PRE_TREATMENT = True

Some mappings

In [38]:
rna_seq_sources = ['Hugo et al.', 'Kwong et al.', 'Yan et al.']
q_pcr_sources = ['Louveau et al.']
micro_array_sources = ['Long et al.', 'Rizos et al.']

source_map = {
    'doi:10.1016/j.cell.2015.07.061': 'Hugo et al.',
    'doi:10.1172/JCI78954DS1': 'Kwong et al.',
    'doi:10.1158/1078-0432.CCR-18-0720': 'Yan et al.',
    'doi:10.3390/cancers11081203': 'Louveau et al.',
    'doi:10.1038/ncomms6694': 'Long et al.',
    'doi:10.1158/1078-0432.CCR-13-3122': 'Rizos et al.'
}

Import GEX data from MelanoDB

In [39]:
import polars as pl

gex = pl.read_csv("../dataset/original/gene_expressions.csv")
gex = gex.with_columns(pl.col('source').replace(source_map))
gex

id,creation_datetime,patientID,sample_id,HGNC,GeneID,description,value,temporality,source
i64,str,str,str,str,str,str,f64,str,str
1,"""2025-04-23 23:02:05.503260""","""LM_1""","""LMSAM_1""","""BRAF""",null,null,7.60798,"""pre treatment""","""Louveau et al."""
2,"""2025-04-23 23:02:05.503285""","""LM_1""","""LMSAM_1""","""RAF1""",null,null,16.095204,"""pre treatment""","""Louveau et al."""
3,"""2025-04-23 23:02:05.503298""","""LM_1""","""LMSAM_1""","""ARAF""",null,null,4.1515,"""pre treatment""","""Louveau et al."""
4,"""2025-04-23 23:02:05.503312""","""LM_1""","""LMSAM_1""","""PDGFRB""",null,null,1.199885,"""pre treatment""","""Louveau et al."""
5,"""2025-04-23 23:02:05.503324""","""LM_1""","""LMSAM_1""","""IGF1R""",null,null,5.47246,"""pre treatment""","""Louveau et al."""
…,…,…,…,…,…,…,…,…,…
8641387,"""2025-04-24 00:42:16.629746""","""HL_Shi-40""","""Pt21-DP2""","""ZYG11A""",null,null,0.019542,"""progression""","""Hugo et al."""
8641388,"""2025-04-24 00:42:16.629757""","""HL_Shi-40""","""Pt21-DP2""","""ZYG11B""",null,null,4.04421,"""progression""","""Hugo et al."""
8641389,"""2025-04-24 00:42:16.629768""","""HL_Shi-40""","""Pt21-DP2""","""ZYX""",null,null,85.7967,"""progression""","""Hugo et al."""


Select only 'pre-treatment', drop QPCR (Louveau) cohort

In [ ]:
if SELECT_PRE_TREATMENT == True:
    gex = gex.filter((pl.col('temporality') == 'pre treatment'))
if DROP_QPCR == True:
    gex = gex.filter(pl.col('source') != 'Louveau et al.')

Drop useless features

In [41]:
gex = gex.drop(['id', 'creation_datetime', 'GeneID', 'description', 'temporality'])

Sample is useless if patientID or HGNC is not given

In [42]:
gex.select(pl.all().null_count())

patientID,sample_id,HGNC,value,source
u32,u32,u32,u32,u32
92181,0,207000,0,0


In [43]:
if DROP_NULLS == True:
    gex = gex.drop_nulls(subset=['patientID', 'HGNC'])
    gex.select(pl.all().null_count())
    
gex.select(pl.all().null_count())

patientID,sample_id,HGNC,value,source
u32,u32,u32,u32,u32
0,0,0,0,0


Add Method column (RNA-seq or Microarray)

In [44]:
gex = gex.with_columns(
    pl.when(pl.col('source').is_in(rna_seq_sources))
    .then(pl.lit('RNA-seq'))
    .when(pl.col('source').is_in(micro_array_sources))
    .then(pl.lit('micro-array'))
    .otherwise(pl.lit('qPCR'))
    .alias('Method')
)
gex

patientID,sample_id,HGNC,value,source,Method
str,str,str,f64,str,str
"""YR_5306""","""03660445B""","""NAT2""",0.0,"""Yan et al.""","""RNA-seq"""
"""YR_5306""","""03660445B""","""ADA""",26.233973,"""Yan et al.""","""RNA-seq"""
"""YR_5306""","""03660445B""","""CDH2""",1.138609,"""Yan et al.""","""RNA-seq"""
"""YR_5306""","""03660445B""","""AKT3""",12.692677,"""Yan et al.""","""RNA-seq"""
"""YR_5306""","""03660445B""","""GAGE12F""",0.0,"""Yan et al.""","""RNA-seq"""
…,…,…,…,…,…
"""HL_Shi-43""","""Pt17-baseline""","""ZYG11A""",0.0625681,"""Hugo et al.""","""RNA-seq"""
"""HL_Shi-43""","""Pt17-baseline""","""ZYG11B""",5.74608,"""Hugo et al.""","""RNA-seq"""
"""HL_Shi-43""","""Pt17-baseline""","""ZYX""",45.907933,"""Hugo et al.""","""RNA-seq"""


Remove duplicate (sample-gene) rows

In [45]:
dupes_pl = (
    gex
    .filter(pl.len().over(['HGNC', 'sample_id']) > 1)
    .sort(['HGNC', 'sample_id'])
)
print(dupes_pl)

shape: (21_042, 6)
┌───────────┬───────────┬────────┬───────┬──────────────┬─────────┐
│ patientID ┆ sample_id ┆ HGNC   ┆ value ┆ source       ┆ Method  │
│ ---       ┆ ---       ┆ ---    ┆ ---   ┆ ---          ┆ ---     │
│ str       ┆ str       ┆ str    ┆ f64   ┆ str          ┆ str     │
╞═══════════╪═══════════╪════════╪═══════╪══════════════╪═════════╡
│ KC_10     ┆ 10A       ┆ ACE    ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_10     ┆ 10A       ┆ ACE    ┆ 2.84  ┆ Kwong et al. ┆ RNA-seq │
│ KC_12     ┆ 12A       ┆ ACE    ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_12     ┆ 12A       ┆ ACE    ┆ 10.27 ┆ Kwong et al. ┆ RNA-seq │
│ KC_13     ┆ 13A       ┆ ACE    ┆ 18.67 ┆ Kwong et al. ┆ RNA-seq │
│ …         ┆ …         ┆ …      ┆ …     ┆ …            ┆ …       │
│ KC_7      ┆ 7A        ┆ mir-95 ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_7      ┆ 7A        ┆ mir-95 ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_9      ┆ 9A        ┆ mir-95 ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_9      ┆ 9A        ┆ mir

In [46]:
gex = gex.sort('value', descending=True).unique(subset=['HGNC', 'sample_id'], keep='first')

dupes_pl = (
    gex
    .filter(pl.len().over(['HGNC', 'sample_id']) > 1)
    .sort(['HGNC', 'sample_id'])
)
print(dupes_pl)

shape: (0, 6)
┌───────────┬───────────┬──────┬───────┬────────┬────────┐
│ patientID ┆ sample_id ┆ HGNC ┆ value ┆ source ┆ Method │
│ ---       ┆ ---       ┆ ---  ┆ ---   ┆ ---    ┆ ---    │
│ str       ┆ str       ┆ str  ┆ f64   ┆ str    ┆ str    │
╞═══════════╪═══════════╪══════╪═══════╪════════╪════════╡
└───────────┴───────────┴──────┴───────┴────────┴────────┘


Save pre-processed GEX

In [47]:
print(gex)
gex.write_csv(f'../dataset/created/gex.csv')

shape: (4_162_394, 6)
┌────────────┬─────────────────┬───────────┬───────────┬──────────────┬─────────────┐
│ patientID  ┆ sample_id       ┆ HGNC      ┆ value     ┆ source       ┆ Method      │
│ ---        ┆ ---             ┆ ---       ┆ ---       ┆ ---          ┆ ---         │
│ str        ┆ str             ┆ str       ┆ f64       ┆ str          ┆ str         │
╞════════════╪═════════════════╪═══════════╪═══════════╪══════════════╪═════════════╡
│ LR_MTP-009 ┆ 28094 PreB      ┆ BCLAF1    ┆ 1207.282  ┆ Long et al.  ┆ micro-array │
│ YR_2220    ┆ 05320216B       ┆ PLA2G4E   ┆ 0.036089  ┆ Yan et al.   ┆ RNA-seq     │
│ HL_Shi-15  ┆ Pt1-baseline    ┆ PPP1R9A   ┆ 0.395961  ┆ Hugo et al.  ┆ RNA-seq     │
│ LR_MTP-034 ┆ 28518_049E PreC ┆ LOC653232 ┆ 11346.65  ┆ Long et al.  ┆ micro-array │
│ KC_16      ┆ 16A             ┆ PMP22     ┆ 49.22     ┆ Kwong et al. ┆ RNA-seq     │
│ …          ┆ …               ┆ …         ┆ …         ┆ …            ┆ …           │
│ RL_WMD-007 ┆ 28067_010A PreB ┆